### Summarizing Text to Avoid Bloat and Help Humans

LLMs can be quite verbose these days! Even as someone who loves to read it gets pretty difficult to read so much text. And if you're working with a paid service or even your own compute, there's no need to spend so much time and energy parsing every token again and again. 

Luckily for us, there's been a lot of work over many years on how to appropriately summarize text. In this notebook, I'll show you a few known strategies, but I'm also excited to hear what works for you and your task.

Additional Reading: 
- [Text Summarization in NLP](https://www.geeksforgeeks.org/nlp/text-summarization-in-nlp/)
- [TextRank as Semantic Search explained](https://www.derwen.ai/docs/ptr/explain_algo/)



### PyTextRank with SpaCy (easy and no token costs!)

Our first strategy is to try a simple and cheap way, informed by early NLP theory. Let's take a look at PyTextRank.


In [9]:
!spacy download en_core_web_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 16.2 MB/s eta 0:00:00m eta 0:00:010:00:01
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')


In [10]:
!pip install pytextrank

In [11]:
!pip install wikipedia

In [12]:
import spacy
nlp = spacy.load("en_core_web_sm")

In [13]:
import wikipedia

In [14]:
page = wikipedia.page("Particle Physics")

In [15]:
page.content

'Particle physics or high-energy physics is the study of fundamental particles and forces that constitute matter and radiation. The field also studies combinations of elementary particles up to the scale of protons and neutrons, while the study of combinations of protons and neutrons is called nuclear physics.\nThe fundamental particles in the universe are classified in the Standard Model as fermions (matter particles) and bosons (force-carrying particles). There are three generations of fermions, although ordinary matter is made only from the first fermion generation. The first generation consists of up and down quarks which form protons and neutrons, and electrons and electron neutrinos. The three fundamental interactions known to be mediated by bosons are electromagnetism, the weak interaction, and the strong interaction.\nQuarks form hadrons, but cannot exist on their own. Hadrons that contain an odd number of quarks are called baryons and those that contain an even number are call

In [16]:
import pytextrank

nlp.add_pipe("textrank", last=True)
doc = nlp(page.content)

/opt/miniconda3/envs/rag/lib/python3.12/site-packages


In [17]:
for p in doc._.phrases:
    print(p.rank, p.count, p.text)

0.09766845826258591 1 Particle physics
0.09766845826258591 4 particle physics
0.09567990509402288 1 such particles
0.0943763326076516 2 fundamental particles
0.09422269530865049 1 Experimental particle physics
0.09368434190694364 2 Theoretical particle physics
0.09368434190694364 3 theoretical particle physics
0.0936052273043467 3 other particles
0.09266317079127025 1 matter particles
0.09236148111509324 1 Particles
0.09236148111509324 1 particle
0.09236148111509324 10 particles
0.09225234872732724 1 Particle accelerators
0.09225234872732724 1 particle accelerators
0.09174752093097285 1 modern particle physics
0.09074998420993369 2 elementary particles
0.09072380459296854 1 particle physicists
0.09041921339108716 1 other particle accelerators
0.08960947120307589 4 composite particles
0.08881612372496851 1 exotic particles
0.0887787999152663 1 Normal particles
0.08875335029367892 1 particle cosmology
0.08859151837717916 1 Subatomic particles
0.08859151837717916 1 subatomic particles
0.0

In [18]:
unit_vector = []

In [19]:
sent_bounds = [ [s.start, s.end, set([])] for s in doc.sents ]

In [20]:
rank_cutoff = 0.08

In [21]:
for idx, p in enumerate(doc._.phrases):
    if p.rank <= rank_cutoff:
        break
    
    unit_vector.append(p.rank)

    for chunk in p.chunks:
        for sent_start, sent_end, sent_vector in sent_bounds:
            if chunk.start >= sent_start and chunk.end <= sent_end:
                sent_vector.add(idx)
                break


In [22]:
unit_vector

[0.09766845826258591,
 0.09766845826258591,
 0.09567990509402288,
 0.0943763326076516,
 0.09422269530865049,
 0.09368434190694364,
 0.09368434190694364,
 0.0936052273043467,
 0.09266317079127025,
 0.09236148111509324,
 0.09236148111509324,
 0.09236148111509324,
 0.09225234872732724,
 0.09225234872732724,
 0.09174752093097285,
 0.09074998420993369,
 0.09072380459296854,
 0.09041921339108716,
 0.08960947120307589,
 0.08881612372496851,
 0.0887787999152663,
 0.08875335029367892,
 0.08859151837717916,
 0.08859151837717916,
 0.08811143504151767,
 0.08792220958508998,
 0.08778598668178612,
 0.08686384956504879,
 0.08639037055869309,
 0.08453531294526298,
 0.08329245527214807,
 0.08309120351895777]

In [23]:
sum_ranks = sum(unit_vector)

unit_vector = [ rank/sum_ranks for rank in unit_vector ]
unit_vector

[0.033683236508373265,
 0.033683236508373265,
 0.03299743775739777,
 0.03254787050564284,
 0.032494885114342315,
 0.03230922143870584,
 0.03230922143870584,
 0.032281936930299106,
 0.03195704685935218,
 0.03185300216677105,
 0.03185300216677105,
 0.03185300216677105,
 0.03181536532788532,
 0.03181536532788532,
 0.03164126373599893,
 0.031297240026622736,
 0.03128821137760316,
 0.031183166026485232,
 0.030903907623979664,
 0.030630303318006414,
 0.030617431335260323,
 0.03060865444210336,
 0.030552842946611033,
 0.030552842946611033,
 0.030387275056767426,
 0.03032201626272439,
 0.030275036630287636,
 0.02995701622586285,
 0.029793725992387974,
 0.02915396628448071,
 0.02872533794638179,
 0.028655931604549212]

In [24]:
from math import sqrt

sent_rank = {}
sent_id = 0

for sent_start, sent_end, sent_vector in sent_bounds:
    sum_sq = 0.0
    for phrase_id in range(len(unit_vector)):
        if phrase_id not in sent_vector:
            sum_sq += unit_vector[phrase_id]**2.0

    sent_rank[sent_id] = sqrt(sum_sq)
    sent_id += 1


In [25]:
sent_rank

{0: 0.17060491532025715,
 1: 0.1741276663666035,
 2: 0.17400778311211365,
 3: 0.17691795111735614,
 4: 0.17691795111735614,
 5: 0.17691795111735614,
 6: 0.17691795111735614,
 7: 0.17691795111735614,
 8: 0.17691795111735614,
 9: 0.17691795111735614,
 10: 0.17402685907792192,
 11: 0.1741481311531303,
 12: 0.17402685907792192,
 13: 0.17691795111735614,
 14: 0.17691795111735614,
 15: 0.17691795111735614,
 16: 0.17691795111735614,
 17: 0.17691795111735614,
 18: 0.17691795111735614,
 19: 0.17691795111735614,
 20: 0.1709731744982882,
 21: 0.17394273666235072,
 22: 0.17691795111735614,
 23: 0.1741276663666035,
 24: 0.17402685907792192,
 25: 0.17691795111735614,
 26: 0.17691795111735614,
 27: 0.17368189602225156,
 28: 0.17402685907792192,
 29: 0.17691795111735614,
 30: 0.17691795111735614,
 31: 0.17691795111735614,
 32: 0.17136488266354333,
 33: 0.1740654815198929,
 34: 0.17691795111735614,
 35: 0.17691795111735614,
 36: 0.17691795111735614,
 37: 0.17402685907792192,
 38: 0.17691795111735614,
 

In [26]:
from operator import itemgetter

sorted(sent_rank.items(), key=itemgetter(1)) 

[(45, 0.15988509939476936),
 (0, 0.17060491532025715),
 (20, 0.1709731744982882),
 (49, 0.17111074134718085),
 (128, 0.17122846115886164),
 (41, 0.1712609008912876),
 (47, 0.1713533226848877),
 (32, 0.17136488266354333),
 (88, 0.17171476571175046),
 (27, 0.17368189602225156),
 (130, 0.17368189602225156),
 (132, 0.17368189602225156),
 (134, 0.17368189602225156),
 (129, 0.17389823907421006),
 (21, 0.17394273666235072),
 (113, 0.17394273666235072),
 (114, 0.17394273666235072),
 (125, 0.17394273666235072),
 (55, 0.17394780244542157),
 (72, 0.17394780244542157),
 (2, 0.17400778311211365),
 (10, 0.17402685907792192),
 (12, 0.17402685907792192),
 (24, 0.17402685907792192),
 (28, 0.17402685907792192),
 (37, 0.17402685907792192),
 (67, 0.17402685907792192),
 (68, 0.17402685907792192),
 (126, 0.17402685907792192),
 (131, 0.17403374372981928),
 (33, 0.1740654815198929),
 (1, 0.1741276663666035),
 (23, 0.1741276663666035),
 (11, 0.1741481311531303),
 (61, 0.1741979044682563),
 (76, 0.1741979044682

In [27]:
limit_sentences = 4

In [28]:
sent_text = {}
sent_id = 0

for sent in doc.sents:
    sent_text[sent_id] = sent.text
    sent_id += 1

In [29]:
num_sent = 0

for sent_id, rank in sorted(sent_rank.items(), key=itemgetter(1)):
    print(sent_id, sent_text[sent_id])
    num_sent += 1
    
    if num_sent == limit_sentences:
        break

45 == Subatomic particles ==

Modern particle physics research is focused on subatomic particles, including atomic constituents, such as electrons, protons, and neutrons (protons and neutrons are composite particles called baryons, made of quarks), that are produced by radioactive and scattering processes; such particles are photons, neutrinos, and muons, as well as a wide range of exotic particles.
0 Particle physics or high-energy physics is the study of fundamental particles and forces that constitute matter and radiation.
20 Experimental particle physics is the study of these particles in radioactive processes and in particle accelerators such as the Large Hadron Collider.
49 Following the convention of particle physicists, the term elementary particles is applied to those particles that are, according to current understanding, presumed to be indivisible and not composed of other particles.





### Summarization using a Local LLM (might work if you prompt tune but takes tokens and cycles!)

Choose your favorite LLM and give it a try. You might need to tune your prompt to ensure it doesn't insert errors or other artifacts that will hinder your workflow (especially if sending the summary to another LLM).

In [30]:
import ollama

In [31]:
prompt = """
You are a summarization assistant. 
You only need to summarize the text, please do not make it longer or add any commentary. 
Find the absolute essential information and distill that.
When possible, use actual terms and sentences from the content, do not rephrase.
"""

In [32]:
messages = [
    { 'role': 'system',
      'content': prompt },
    { 'role': 'user',
      'content': "Please summarize the following: {}".format(page.content),
    }
]

In [33]:
response = ollama.chat(
    model="gemma3:1b",
    messages=messages)

In [34]:
response.message.content

'Particle physics or high-energy physics is the study of fundamental particles and forces that constitute matter and radiation. The field also studies combinations of elementary particles up to the scale of protons and neutrons, while the study of combinations of protons and neutrons is called nuclear physics.\n\nThe fundamental particles in the universe are classified in the Standard Model as fermions (matter particles) and bosons (force-carrying particles). There are three generations of fermions, although ordinary matter is made only from the first fermion generation. The first generation consists of up and down quarks which form protons and neutrons, and electrons and electron neutrinos. The three fundamental interactions known to be mediated by bosons are electromagnetism, the weak interaction, and the strong interaction.\n\nQuarks form hadrons, but cannot exist on their own. Hadrons that contain an odd number of quarks are called baryons and those that contain an even number are 

### Summarization using a task-specifc deep learning model (Advanced Mode)



In [ ]:
# Use a pipeline as a high-level helper
# Warning: Pipeline type "summarization" is no longer supported in transformers v5.
# You must load the model directly (see below) or downgrade to v4.x with:
# 'pip install "transformers<5.0.0'

# see https://www.w3tutorials.net/blog/how-to-truncate-input-in-the-huggingface-pipeline/ for more information on chunking for other pipelines

In [ ]:
from transformers import AutoTokenizer, pipeline

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("google/pegasus-xsum")
pipe = pipeline("summarization", model="google/pegasus-xsum")

In [ ]:
tokenized = tokenizer(
    page.content,  
    truncation=True,  
    max_length=512,  
    return_overflowing_tokens=True,  # Split into chunks  
    stride=24,  # Overlap 24 tokens between chunks (preserve context)  
    return_offsets_mapping=False  
) 

In [ ]:
chunks = tokenized["input_ids"]
results = []

In [ ]:
for chunk in chunks:
    chunk_text = tokenizer.decode(chunk, skip_special_tokens=True) # turn back into text to run in pipeline
    summ_results = pipe(chunk_text)
    results.extend(summ_results)

In [ ]:
results

What worked best for you? Any other strategies you have found and want to share? 